In [6]:
import torch
import sys, os, pdb
import argparse, logging
import torch.nn.functional as F

from pathlib import Path

import librosa

from src.model.age_sex.wavlm_demographics import WavLMWrapper

In [2]:
labels = ["Female", "Male"]
device = "cpu"

In [3]:
# Define the model
# Note that ensemble yields the better performance than the single model
wavlm_model = WavLMWrapper.from_pretrained("tiantiaf/wavlm-large-age-sex")
_ = wavlm_model.eval().to(device)

In [18]:
# Our training data filters output audio shorter than 3 seconds (unreliable predictions) and longer than 15 seconds (computation limitation)
# So you need to prepare your audio to a maximum of 15 seconds, 16kHz and mono channel
data, _ = librosa.load("/workspace/decoding/assets/misc/LJ037-0171.wav", sr=16000)
data = torch.tensor(data)[None]
wavlm_age_outputs, wavlm_sex_outputs = wavlm_model(data)

# Age is between 0-100
age_pred = wavlm_age_outputs.detach().cpu().numpy()

sex_prob = F.softmax(wavlm_sex_outputs, dim=1)
print(labels[torch.argmax(sex_prob).detach().cpu().item()])
print(sex_prob)

Female
tensor([[1.0000e+00, 3.7581e-06]], grad_fn=<SoftmaxBackward0>)
